# Pronóstico de Inflación Subyacente en México con ARIMA

## Objetivo
Construir un modelo ARIMA para pronosticar la inflación subyacente mensual en México,
utilizando datos históricos del Sistema de Información Económica (SIE) del Banco de México.

## Justificación
La inflación subyacente excluye los componentes volátiles del INPC (energéticos y 
agropecuarios), por lo que refleja las presiones de demanda estructurales de la economía 
y es el indicador de referencia para las decisiones de política monetaria del Banxico.
Un modelo ARIMA es apropiado porque la serie presenta autocorrelación significativa 
y una dinámica de persistencia consistente con procesos autorregresivos.

## Fuente de datos
Banco de México — SIE, Serie SF43783 (Índice de precios subyacente, variación mensual)

## Herramientas
Python · pandas · statsmodels · matplotlib · plotly

In [1]:
# --- Librerías ---
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficas
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Librerías cargadas correctamente ✓")

Librerías cargadas correctamente ✓


## 1. Obtención de datos

Conectamos directamente a la API del SIE de Banxico para descargar la serie de 
inflación subyacente mensual (SF43783). Esta serie representa la variación mensual 
del índice de precios subyacente, que excluye energéticos y agropecuarios.

Para replicar este notebook se requiere un token de acceso gratuito del SIE Banxico,
disponible en: https://www.banxico.org.mx/SieAPIRest/service/v1/

In [4]:
# --- Conexión a la API del SIE Banxico ---
TOKEN = "4fa7f18578436202fb8a51db8c973e6de62f0b51c296600d2b9216f4f5af8d7f"

# SP74660: Inflación subyacente mensual (%) — fuente directa INEGI/Banxico
SERIE = "SP74660"
URL = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{SERIE}/datos"

headers = {"Bmx-Token": TOKEN}
response = requests.get(URL, headers=headers)
data = response.json()

# Convertir a DataFrame
registros = data["bmx"]["series"][0]["datos"]
df = pd.DataFrame(registros)
df.columns = ["fecha", "inflacion"]
df["fecha"] = pd.to_datetime(df["fecha"], format="%d/%m/%Y")
df["inflacion"] = pd.to_numeric(df["inflacion"], errors="coerce")
df = df.dropna().set_index("fecha").sort_index()

print(f"Serie cargada: {len(df)} observaciones")
print(f"Periodo: {df.index[0].strftime('%b %Y')} — {df.index[-1].strftime('%b %Y')}")
print(f"\nEstadísticas:")
print(df["inflacion"].describe().round(4))
df.tail()

Serie cargada: 531 observaciones
Periodo: Feb 1982 — Apr 2026

Estadísticas:
count    531.0000
mean       1.4152
std        2.0852
min       -0.0800
25%        0.2900
50%        0.4800
75%        1.3850
max       13.2100
Name: inflacion, dtype: float64


,inflacion
fecha,
2025-12-01,0.41
2026-01-01,0.60
2026-02-01,0.46
2026-03-01,0.38
2026-04-01,0.31
